# Fetch model weights

Downloads a weights archive from the authors' HuggingFace dataset
(`bgao95/DrugCLIP_data`) into `model/`, which is gitignored.

### Which weights

| checkpoint | what it is | when to use |
| --- | --- | --- |
| `benchmark_weights/dude_identity_90.pt` | single model, 90% identity filtered | the paper's **benchmark** setting; fine for getting a pipeline working |
| `model_weights.zip` -> `6_folds/` | **6-model ensemble** | the general-purpose choice — what the paper used for NET, TRIP12 and the genome-wide screen |
| `model_weights.zip` -> `8_folds/` | 8-model ensemble with extra augmentation | 5HT2AR only |

From the Science methods:

> We use an ensemble model for most applications unless stated otherwise,
> including wet-lab validations with the NET and TRIP12 target and the final
> genome-wide virtual screening. These applications follow a six-fold
> cross-validation strategy

and separately:

> For the 5HT2AR target, we adopt an eight-fold cross-validation strategy and
> apply data augmentation techniques

The folds are **training-data splits, not targets** — DrugCLIP is zero-shot, so
no checkpoint is tuned for any particular protein. The ensemble reduces variance
rather than specialising.

`model_weights.zip` is 13 GB and unpacks to 16.7 GB; the 6-fold set alone is
7.2 GB. The archive also has a sibling, `encoded_mol_embs.zip`, which holds
pre-encoded **ChemDiv** molecule embeddings rather than weights — useful only if
screening that library.

## Configuration

In [ ]:
from pathlib import Path
import shutil, zipfile, urllib.request, collections, subprocess, sys

REPO    = Path("/root/GitRepos/DrugCLIP")
MODEL   = REPO / "model"                  # gitignored
ARCHIVE = "model_weights.zip"
REPO_ID = "bgao95/DrugCLIP_data"

# which subdirectories to extract; [] means everything
EXTRACT_ONLY = ["6_folds"]                # ["6_folds", "8_folds"] for both
DELETE_ARCHIVE_AFTER = True               # reclaim 13 GB once extraction is verified

URL  = f"https://huggingface.co/datasets/{REPO_ID}/resolve/main/{ARCHIVE}"
DEST = MODEL / ARCHIVE
MODEL.mkdir(parents=True, exist_ok=True)
print("destination:", DEST)

## Download

Resumable — rerun this cell to continue an interrupted download rather than
starting again.

The 13 GB takes a while, so progress goes to a log file; `wget`'s progress bar
renders poorly inside a notebook cell.

In [ ]:
log = MODEL / "download.log"

if DEST.exists():
    print(f"already present: {DEST.stat().st_size / 1e9:.2f} GB")

r = subprocess.run(["wget", "-c", "--progress=dot:giga", "-O", str(DEST), URL],
                   stderr=open(log, "w"), stdout=subprocess.DEVNULL)
print("exit:", r.returncode, f"| size: {DEST.stat().st_size / 1e9:.2f} GB")

## Verify and inspect before extracting

A truncated download still produces a file, so check the archive is intact and
see how large it unpacks to **before** committing disk to it.

In [ ]:
assert zipfile.is_zipfile(DEST), (
    "not a valid zip - the download is incomplete; rerun the cell above to resume")

with zipfile.ZipFile(DEST) as z:
    assert z.testzip() is None, "archive contains a corrupt member"
    members = [m for m in z.infolist() if not m.is_dir()]

unpacked = sum(m.file_size for m in members)
free = shutil.disk_usage(MODEL).free
print(f"files            : {len(members)}")
print(f"unpacked size    : {unpacked / 1e9:.2f} GB")
print(f"free on this disk: {free / 1e9:.0f} GB")
assert free > unpacked * 1.1, "not enough free space to extract"

groups = collections.Counter()
for m in members:
    groups[m.filename.split("/")[1] if "/" in m.filename else "(root)"] += m.file_size
for k, v in sorted(groups.items()):
    mark = "  <-- selected" if (not EXTRACT_ONLY or k in EXTRACT_ONLY) else ""
    print(f"  {k:<16} {v / 1e9:6.2f} GB{mark}")

## Extract

Only the groups in `EXTRACT_ONLY` are unpacked, so the 5HT2AR-specific
`8_folds` set never reaches disk unless asked for.

The archive wraps everything in a `model_weights/` directory, which is stripped
so the result is simply:

```
model/
  6_folds/
    fold_0.pt ... fold_5.pt
```

In [ ]:
with zipfile.ZipFile(DEST) as z:
    allm = [m for m in z.infolist() if not m.is_dir()]
    wanted = [m for m in allm
              if not EXTRACT_ONLY or any(f"/{k}/" in m.filename for k in EXTRACT_ONLY)]
    print(f"extracting {len(wanted)} of {len(allm)} files "
          f"({sum(m.file_size for m in wanted) / 1e9:.1f} GB)")

    for m in wanted:
        # strip the leading "model_weights/" so we get model/6_folds/... directly
        rel = Path(*Path(m.filename).parts[1:])
        target = MODEL / rel
        target.parent.mkdir(parents=True, exist_ok=True)
        with z.open(m) as src, open(target, "wb") as dst:
            shutil.copyfileobj(src, dst)

folds = sorted(f for k in (EXTRACT_ONLY or ["6_folds", "8_folds"])
               for f in (MODEL / k).glob("*.pt"))
print(f"\n{len(folds)} checkpoints:")
for f in folds:
    print(f"  {f.stat().st_size / 1e9:5.2f} GB  {f.relative_to(MODEL)}")

## Verify, then clean up

Deleting the archive means a 13 GB re-download if something is wrong, so the
extracted files are checked first: the expected number of checkpoints, each a
plausible size, and the first one actually loadable by torch.

Nothing is removed unless every check passes.

In [ ]:
import torch

EXPECTED = {"6_folds": 6, "8_folds": 8}
ok, problems = True, []

for group in (EXTRACT_ONLY or list(EXPECTED)):
    d = MODEL / group
    got = sorted(d.glob("*.pt")) if d.exists() else []
    want = EXPECTED.get(group)
    if want and len(got) != want:
        problems.append(f"{group}: found {len(got)} checkpoints, expected {want}")
        ok = False
    for f in got:
        gb = f.stat().st_size / 1e9
        if not 0.9 < gb < 2.0:
            problems.append(f"{f.name}: implausible size {gb:.2f} GB")
            ok = False

# size alone does not prove a file is readable - load one for real
if ok and folds:
    try:
        sd = torch.load(folds[0], map_location="cpu", weights_only=False)
        print(f"loaded {folds[0].name}: {len(sd.get('model', sd))} tensors - readable")
    except Exception as e:
        problems.append(f"torch.load failed on {folds[0].name}: {e}")
        ok = False

print()
print("all checks passed" if ok else "PROBLEMS - nothing will be deleted:")
for pr in problems:
    print("  -", pr)

In [ ]:
if not ok:
    print("checks did not pass - skipping cleanup")
else:
    freed = 0

    # the now-empty wrapper left behind by the archive
    wrapper = MODEL / Path(ARCHIVE).stem
    if wrapper.exists():
        shutil.rmtree(wrapper)
        print(f"removed empty {wrapper.name}/")

    # any group that was extracted but not asked for
    for d in sorted(MODEL.iterdir()):
        if d.is_dir() and EXTRACT_ONLY and d.name in EXPECTED and d.name not in EXTRACT_ONLY:
            n = sum(f.stat().st_size for f in d.rglob("*") if f.is_file())
            shutil.rmtree(d)
            freed += n
            print(f"removed {d.name}/ ({n / 1e9:.1f} GB)")

    if DELETE_ARCHIVE_AFTER and DEST.exists():
        n = DEST.stat().st_size
        DEST.unlink()
        freed += n
        print(f"removed {DEST.name} ({n / 1e9:.1f} GB)")

    log = MODEL / "download.log"
    if log.exists():
        log.unlink()

    print(f"\nreclaimed {freed / 1e9:.1f} GB")

print("\n--- model/ now holds ---")
for f in sorted(MODEL.rglob("*")):
    if f.is_file():
        print(f"  {f.stat().st_size / 1e9:6.2f} GB  {f.relative_to(MODEL)}")
print(f"\nfree on this disk: {shutil.disk_usage(MODEL).free / 1e9:.0f} GB")

## Using the ensemble

`retrieval.py` takes a single `--path`, so it cannot load six checkpoints at
once. The paper combines them by **mean pooling** the predictions:

> these six model weights are used to generate six different predictions, which
> are then combined using mean pooling

In practice that means screening once per fold and averaging the scores. Note
this is an average **across folds**, which is separate from the per-pocket
z-score normalisation used when combining several pockets — the two solve
different problems and both apply when you have several of each.

Set `WEIGHTS` in a screening notebook to one of these checkpoints to use a
single fold, which is a reasonable starting point:

```python
WEIGHTS = REPO / "model" / "6_folds" / "fold_0.pt"
```